# 02a｜首批真实语料的 24M 预训练试跑

130U-FinLLM｜130U 金融大语言模型

从随机初始化开始，使用 seed_v1 的 8K tokenizer 和 Wikipedia 分片训练 61 次更新（约 1M tokens）。先完成 01a。本页只执行通用预训练的短段，不会自动进入 100M 长训练、SFT 或 DPO。

每个 Notebook 都包含独立环境入口。Colab 菜单：运行时 → 更改运行时类型 → T4 GPU。点击执行代码前可查看源码；所有输出均为实际执行后产生，仓库不预填训练成绩。

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
command(sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<3", "tokenizers>=0.21,<0.23", "scikit-learn>=1.5,<2")
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps")

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")

In [ ]:
SEED_VERSION = "seed_v1"
SEED_DRIVE = DRIVE_ROOT / "seeds" / SEED_VERSION
# 独立版本快照不会改写其他版本或小实验的备份清单。
for name, relative, required in (
    ("processed", f"data/processed/{SEED_VERSION}", "seed_summary.json"),
    ("tokenizer", f"tokenizer/{SEED_VERSION}", "tokenizer.json")):
    destination = PROJECT / relative
    if not (destination / required).is_file():
        require(SEED_DRIVE / name / "SNAPSHOT_COMPLETE.json")
        finllm("sync", "--source", SEED_DRIVE / name,
               "--destination", destination, "--working-copy")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("请选择 T4 GPU 运行时。")
CONFIG = Path("data/processed") / SEED_VERSION / "pilot_config.json"
require(CONFIG)
runtime_config = json.loads(CONFIG.read_text(encoding="utf-8"))
runtime_config["backup_dir"] = str(DRIVE_ROOT / "live_checkpoints" / (SEED_VERSION + "_pretrain"))
runtime_path = Path("configs/runtime") / (SEED_VERSION + "_pilot.json")
runtime_path.parent.mkdir(parents=True, exist_ok=True)
runtime_path.write_text(json.dumps(runtime_config, indent=2), encoding="utf-8")
print(json.dumps(runtime_config, indent=2))
print("GPU:", torch.cuda.get_device_name(0))

运行下面单元格开始短段训练。已有本版本检查点时自动校验并恢复最新进度；数据或运行时不兼容会明确报错，不覆盖进度重新开始。

In [ ]:
from finllm.runtime import restore_seed_checkpoint
RESUME_CHECKPOINT = restore_seed_checkpoint(runtime_config["output_dir"], runtime_config["backup_dir"])
arguments = ["train", "--config", runtime_path]
if RESUME_CHECKPOINT:
    arguments += ["--resume", RESUME_CHECKPOINT]
finllm(*arguments)
print((Path(runtime_config["output_dir"]) / "run_summary.json").read_text(encoding="utf-8"))

In [ ]:
SEED_DRIVE = DRIVE_ROOT / "seeds" / SEED_VERSION
for name, source in (("raw", Path("data/raw") / SEED_VERSION),
                     ("processed", Path("data/processed") / SEED_VERSION),
                     ("tokenizer", Path("tokenizer") / SEED_VERSION),
                     ("run", Path("runs") / (SEED_VERSION + "_pretrain"))):
    if source.exists():
        finllm("sync", "--source", PROJECT / source, "--destination", SEED_DRIVE / name)
shutil.copy2(PROJECT / "runtime_revision.txt", SEED_DRIVE / "runtime_revision.txt")
print("本版本备份完成：", SEED_DRIVE)

将上方显示的 GPU 型号及 run_summary.json 中的峰值显存、tokens/second、tokens 和验证 loss 带回工作对话后决定下一段预算。SEntFiN 的立场审核和资产映射仍是 SFT/DPO 的前置材料；本次不声称金融预测能力。